# UPI Transaction Analysis - Data Cleaning and Preparation

## Objective
The objective of this notebook is to clean and prepare UPI transaction data for further analysis in SQL and Power BI.

## Workflow
- Load raw UPI transaction dataset
- Clean column names and data types
- Handle formatting issues
- Create normalized tables for SQL
- Export cleaned and structured CSV files

## 1. Loading the Dataset

In this step, the raw UPI transaction dataset is loaded into Python using Pandas for preprocessing and transformation.

In [1]:
import pandas as pd

df = pd.read_csv('upi_transactions_2024.csv')
df.head()

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0


In [2]:
df.columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype
---  ------              --------------   -----
 0   transaction id      250000 non-null  str  
 1   timestamp           250000 non-null  str  
 2   transaction type    250000 non-null  str  
 3   merchant_category   250000 non-null  str  
 4   amount (INR)        250000 non-null  int64
 5   transaction_status  250000 non-null  str  
 6   sender_age_group    250000 non-null  str  
 7   receiver_age_group  250000 non-null  str  
 8   sender_state        250000 non-null  str  
 9   sender_bank         250000 non-null  str  
 10  receiver_bank       250000 non-null  str  
 11  device_type         250000 non-null  str  
 12  network_type        250000 non-null  str  
 13  fraud_flag          250000 non-null  int64
 14  hour_of_day         250000 non-null  int64
 15  day_of_week         250000 non-null  str  
 16  is_weekend          250000 non-

## 2. Standardizing Column Names

The original dataset contained column names with spaces, brackets, and inconsistent formatting.  
To make the data easier to work with in Python, SQL, and Power BI, the column names were cleaned and standardized by:
- converting them to lowercase
- replacing spaces with underscores
- removing special characters such as brackets

In [4]:
df.columns = [
    'transaction_id', 'timestamp', 'transaction_type', 'merchant_category',
    'amount', 'transaction_status', 'sender_age_group', 'receiver_age_group',
    'sender_state', 'sender_bank', 'receiver_bank', 'device_type',
    'network_type', 'fraud_flag', 'hour_of_day', 'day_of_week', 'is_weekend'
]

## 3. Data Type Conversion

The timestamp column was converted into a proper datetime format to enable time-based analysis.  
This helps in extracting time-related insights such as hourly trends and day-wise transaction behavior.

In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [7]:
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df.head()

,transaction_id,timestamp,transaction_type,merchant_category,amount,transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,year,month
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0,2024,10
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0,2024,4
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0,2024,4
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1,2024,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0,2024,1


In [8]:
df.to_csv('cleaned_upi_data.csv', index=False)

## 4. Preparing Normalized Tables

To support advanced SQL analysis, the dataset is transformed into multiple related tables instead of keeping everything in a single flat file.

The following tables were created:
- **users**: sender demographic and banking information
- **receivers**: receiver demographic and banking information
- **transactions**: core transaction details such as amount, status, fraud flag, and IDs
- **transaction_details**: transaction behavior and contextual information such as category, device type, network type, hour, and weekday

This normalization makes the data model more realistic and allows joins, subqueries, CTEs, views, and stored procedures to be demonstrated in SQL.

In [9]:
# CREATING A USERS TABLE

users = df[['sender_age_group', 'sender_state', 'sender_bank']].drop_duplicates().reset_index(drop=True)
users['user_id'] = users.index + 1

users.head()

,sender_age_group,sender_state,sender_bank,user_id
0,26-35,Delhi,Axis,1
1,26-35,Uttar Pradesh,ICICI,2
2,26-35,Karnataka,Yes Bank,3
3,26-35,Delhi,ICICI,4
4,36-45,Karnataka,IndusInd,5


In [10]:
# CREATING A RECEIVERS TABLE

receivers = df[['receiver_age_group', 'receiver_bank']].drop_duplicates().reset_index(drop=True)
receivers['receiver_id'] = receivers.index + 1

receivers.head()

,receiver_age_group,receiver_bank,receiver_id
0,18-25,SBI,1
1,26-35,Axis,2
2,36-45,PNB,3
3,26-35,PNB,4
4,18-25,Yes Bank,5


In [12]:
# CREATING A TRANSACTIONS TABLE

transactions = df[['transaction_id', 'timestamp', 'amount',
                   'transaction_status', 'fraud_flag',
                   'user_id', 'receiver_id']]

transactions.head()

,transaction_id,timestamp,amount,transaction_status,fraud_flag,user_id,receiver_id
0,TXN0000000001,2024-10-08 15:17:28,868,SUCCESS,0,1,1
1,TXN0000000002,2024-04-11 06:56:00,1011,SUCCESS,0,2,2
2,TXN0000000003,2024-04-02 13:27:18,477,SUCCESS,0,3,3
3,TXN0000000004,2024-01-07 10:09:17,2784,SUCCESS,0,4,4
4,TXN0000000005,2024-01-23 19:04:23,990,SUCCESS,0,1,5


## 5. Mapping User and Receiver IDs

Unique IDs are assigned to users and receivers so that the tables could be linked properly through relational keys.  
These IDs are then mapped back into the main transaction data to support structured database design.

In [ ]:
# MAPPING ID'S BACK TO MAIN DATAFRAME

df = df.merge(users, on=['sender_age_group', 'sender_state', 'sender_bank'])
df = df.merge(receivers, on=['receiver_age_group', 'receiver_bank'])

df.head()

## 6. Exporting Cleaned Files

The cleaned and transformed tables were exported as CSV files for further use in:
- **MySQL Workbench** for SQL analysis
- **Power BI** for dashboard creation

Exported files:
- `users.csv`
- `receivers.csv`
- `transactions_small.csv`
- `transaction_details_small.csv`

In [13]:
# SAVING NEW FILES TO CSV

users.to_csv('users.csv', index=False)
receivers.to_csv('receivers.csv', index=False)
transactions.to_csv('transactions.csv', index=False)
transactions.to_csv('transaction_details.csv', index=False)

## 7. Creating a Reduced Dataset

The original transaction table was large for direct import into SQL Workbench.  
To make the workflow manageable and avoid import issues, a reduced sample of the dataset was created.

This reduced dataset preserves the structure and analytical value of the original data while making it easier to work with in SQL and Power BI.

In [ ]:
# Creating a small dataset from the large original dataset of transactions

import pandas as pd

df = pd.read_csv('transactions.csv')

# take only 2000 rows (safe size)
df_small = df.sample(n=2000, random_state=42)

df_small.to_csv('transactions_small.csv', index=False)

print("Small dataset created")

Small dataset created


## 8. Creating a Reduced Dataset

The original transactions_details table was large for direct import into SQL Workbench.  
To make the workflow manageable and avoid import issues, a reduced sample of the dataset was created.

This reduced dataset preserves the structure and analytical value of the original data while making it easier to work with in SQL and Power BI.

In [ ]:
# Creating a small dataset from the large original dataset of transactions_details

import pandas as pd

# load ORIGINAL dataset (not reduced one)
df = pd.read_csv('upi_transactions_2024.csv')   # use your actual file name

# clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
)

# IMPORTANT: filter only those transactions already in SQL
transactions_sql = pd.read_csv('transactions_small.csv')

# keep only matching transaction_ids
df_filtered = df[df['transaction_id'].isin(transactions_sql['transaction_id'])]

# now create details file
details_small = df_filtered[['transaction_id', 'transaction_type',
                            'merchant_category', 'device_type',
                            'network_type', 'hour_of_day',
                            'day_of_week', 'is_weekend']]

details_small.to_csv('transaction_details_small.csv', index=False)

print("Details file created correctly!")

Details file created correctly!


## 9. Summary

Using Python, the raw UPI transaction dataset was cleaned, standardized, and transformed into a structured format suitable for relational analysis.

### Key tasks completed:
- Loaded and inspected raw transaction data
- Cleaned column names
- Converted data types
- Created a reduced dataset for manageable SQL import
- Built normalized tables for advanced SQL operations
- Exported cleaned files for SQL and Power BI

This preprocessing step formed the foundation for the next stages of the project:
1. SQL-based analytical querying
2. Power BI dashboard development

## Why Python was used in this project

Python was used as the preprocessing layer of the project because it is efficient for:
- cleaning raw data
- standardizing columns
- transforming large flat files
- preparing normalized tables
- exporting structured files for downstream tools

This makes the dataset easier to analyze in SQL and visualize in Power BI.